# DL Text Tokenization and Sequencing  

To effectively train **Deep Learning models** for text classification, raw textual data must be transformed into a structured numerical format that neural networks can process. Unlike **traditional Machine Learning models** that leverage **TF-IDF** or **Bag-of-Words (BoW)** for feature extraction, deep learning architectures rely on **embedding-based representations** to capture semantic relationships between words.  

In this notebook, we focus on **preparing text data for deep learning models** by performing the following key steps:  

- **Tokenization**: Converting text into a vocabulary of indexed tokens using Keras' `Tokenizer`.  
- **Sequence Encoding**: Mapping words to their corresponding integer representations.  
- **Padding Sequences**: Ensuring uniform input size by applying zero-padding to sequences.  

These transformations are crucial for feeding textual data into **Recurrent Neural Networks (RNNs)**, **Long Short-Term Memory Networks (LSTMs)**, **Gated Recurrent Units (GRUs)**, **1D Convolutional Neural Networks (Conv1D)**, and **Deep Neural Networks (DNNs)**. Each of these architectures expects fixed-length input sequences and benefits from structured token representations.  




## 1. Import Required Libraries 

In [6]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import sys
import os
from pathlib import Path
import importlib
import pickle
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


# Get the current notebook directory
CURRENT_DIR = Path(os.getcwd()).resolve()

# Automatically find the project root (go up 1 level)
PROJECT_ROOT = CURRENT_DIR.parents[1]

# Add project root to sys.path
sys.path.append(str(PROJECT_ROOT))

# Function to get relative paths from project root
def get_relative_path(absolute_path):
    return str(Path(absolute_path).relative_to(PROJECT_ROOT))

# Print project root directory
print(f"Project Root Directory: {PROJECT_ROOT.name}")  # Display only the root folder name

import config  # Now Python can find config.py

## 2. Load Preprocessed Data

We now **load the preprocessed datasets** to prepare them for Deep Learning:  
✔ **`X_train_split.pkl`** → Training dataset (80%).  
✔ **`X_val_split.pkl`** → Validation dataset (20%).  
✔ **`X_test_sub_cleaned_final.pkl`** → Test dataset for submission (no labels). 


In [5]:
# Reload config to ensure any updates are applied
importlib.reload(config)  

# Define paths for datasets
data_dir = Path(config.PROCESSED_DIR)
train_path = data_dir / "X_train_split.pkl"
val_path = data_dir / "X_val_split.pkl"
test_sub_path = data_dir / "X_test_sub_cleaned_final.pkl"

# Function to load a Pickle file safely
def load_pickle(file_path, dataset_name):
    """Loads a pickle file with error handling and basic visualization."""
    if not file_path.exists():
        print(f"[X] Error: `{dataset_name}` file not found at {file_path}")
        return None

    try:
        data = pd.read_pickle(file_path)
        print(f"[✔] Successfully loaded `{dataset_name}` | Shape: {data.shape}")

        if isinstance(data, pd.DataFrame) and not data.empty:
            display(data.head())  # Display first rows for quick verification

        return data
    except Exception as e:
        print(f"[X] Error loading `{dataset_name}`: {e}")
        return None

# Load datasets
X_train = load_pickle(train_path, "X_train_split.pkl")
X_val = load_pickle(val_path, "X_val_split.pkl")
X_test_sub = load_pickle(test_sub_path, "X_test_sub_cleaned_final.pkl")


[✔] Successfully loaded `X_train_split.pkl` | Shape: (67932, 9)


,designation,description,text,productid,imageid,prdtypecode,prdtypecode_encoded,Label,image_name
1887,porte bebe violet rouge trois mere multifoncti...,Porte bébé Violet et rouge Trois-en-un mère mu...,porte bebe violet rouge trois mere multifoncti...,3050424970,1187504001,1320,12,Early Childhood,image_1187504001_product_3050424970.jpg
70389,jesus cahiers libre avenir,Prêtre autrement.,jesus cahiers libre avenir pretre autrement,131641431,885888766,10,0,Adult Books,image_885888766_product_131641431.jpg
59835,chambre paillasson forme coeur tapis fluffy ta...,Chambre Paillasson en forme de coeur Tapis Tap...,chambre paillasson forme coeur tapis fluffy ch...,4197486437,1313030973,1560,13,Interior Furniture and Bedding,image_1313030973_product_4197486437.jpg
23220,pcs alliage aluminium portail carter entrainem...,2pcs en alliage d&#39;aluminium Portail du car...,pcs alliage aluminium portail carter entrainem...,3929174950,1265009801,1280,7,Toys for Children,image_1265009801_product_3929174950.jpg
36107,harnais chien arnais noir anti traction gilet ...,<p><b>La description:</b></p><br /><p> Fait de...,harnais chien arnais noir anti traction gilet ...,4183293159,1313455838,2220,17,Supplies for Domestic Animals,image_1313455838_product_4183293159.jpg


[✔] Successfully loaded `X_val_split.pkl` | Shape: (16984, 9)


,designation,description,text,productid,imageid,prdtypecode,prdtypecode_encoded,Label,image_name
81432,bas filles enfants enfants collant coton bebe ...,Filles Bas Enfants Enfants Collant Coton bébé ...,bas filles enfants collant coton bebe stocking...,3898715946,1261369347,1301,10,Accessories for Children,image_1261369347_product_3898715946.jpg
44734,cosmic planete series peluche capuche couvertu...,Cosmic Planète Series en peluche avec capuche ...,cosmic planete series peluche capuche couvertu...,4205111198,1315322348,1560,13,Interior Furniture and Bedding,image_1315322348_product_4205111198.jpg
59366,dolphin robot electrique piscine fond parois l...,dolphin dolphin - robot électrique de piscine ...,dolphin robot electrique piscine fond parois l...,3894338575,1260564839,2583,23,Piscine and Spa,image_1260564839_product_3894338575.jpg
36932,haydaim pokemon noir blanc,NaN,haydaim pokemon noir blanc,155433978,911177853,1160,5,Playing Cards,image_911177853_product_155433978.jpg
69999,lot livres partitions piano bach busoni clavie...,NaN,lot livres partitions piano bach busoni clavie...,2145087508,1128429580,2403,19,Children Books and Magazines,image_1128429580_product_2145087508.jpg


[✔] Successfully loaded `X_test_sub_cleaned_final.pkl` | Shape: (13812, 6)


,designation,description,text,productid,imageid,image_name
84916,folkmanis puppets marionnette theatre mini turtle,NaN,folkmanis puppets marionnette theatre mini turtle,516376098,1019294171,image_1019294171_product_516376098.jpg
84917,porte flamme gaxix flamebringer gaxix twilight...,NaN,porte flamme gaxix flamebringer twilight dragons,133389013,1274228667,image_1274228667_product_133389013.jpg
84918,pompe filtration speck badu,NaN,pompe filtration speck badu,4128438366,1295960357,image_1295960357_product_4128438366.jpg
84919,robot piscine electrique,<p>Ce robot de piscine d&#39;un design innovan...,robot piscine electrique design innovant elega...,3929899732,1265224052,image_1265224052_product_3929899732.jpg
84920,hsm destructeur securio coupe croise,NaN,hsm destructeur securio coupe croise,152993898,940543690,image_940543690_product_152993898.jpg


## 3. Apply Tokenization and Sequencing  

To prepare text data for Deep Learning models, we:  
✔ **Tokenize the text** → Convert words into integer indices based on frequency.  
✔ **Apply padding** → Ensure all sequences have the same length for efficient batch processing.  

###  **Key Parameters**  
- **`MAX_VOCAB_SIZE = 20,000`** → Limits the vocabulary to the most frequent words.  
- **`maxlen = 500`** → Truncates or pads sequences to ensure a uniform length. 


In [7]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Extract text data
train_text = X_train["text"]
val_text = X_val["text"]
test_text = X_test_sub["text"]

# Define hyperparameters
MAX_VOCAB_SIZE = 20000  # Maximum number of words in the vocabulary
maxlen = 500  # Maximum sequence length

# Initialize and fit the tokenizer on training text
tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(train_text)

# Convert text to sequences
X_train_seq = tokenizer.texts_to_sequences(train_text)
X_val_seq = tokenizer.texts_to_sequences(val_text)
X_test_seq = tokenizer.texts_to_sequences(test_text)

# Apply padding to ensure uniform sequence length
X_train_pad = pad_sequences(X_train_seq, maxlen=maxlen, padding="post", truncating="post")
X_val_pad = pad_sequences(X_val_seq, maxlen=maxlen, padding="post", truncating="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=maxlen, padding="post", truncating="post")

# Print shape of the processed datasets
print(f"[✔] Tokenization & Padding completed")
print(f" X_train_pad shape: {X_train_pad.shape}")
print(f" X_val_pad shape: {X_val_pad.shape}")
print(f" X_test_pad shape: {X_test_pad.shape}")


[✔] Tokenization & Padding completed
 X_train_pad shape: (67932, 500)
 X_val_pad shape: (16984, 500)
 X_test_pad shape: (13812, 500)


## 4. Save Processed Data

Now that we have **tokenized and padded our text data**, we save the processed files for future use in Deep Learning models.  

### **Saved Files & Their Purpose**  
✔ **`X_train_pad_dl.pkl`** → Tokenized and padded training dataset.  
✔ **`X_val_pad_dl.pkl`** → Tokenized and padded validation dataset.  
✔ **`X_test_sub_pad_dl.pkl`** → Tokenized and padded test dataset (for final submission).  
✔ **`tokenizer_dl.pkl`** → The trained tokenizer, ensuring consistent preprocessing during model inference.  



In [10]:
# Reload config to ensure any updates are applied
importlib.reload(config) 

# # Define save paths in the processed directory
# save_dir = Path(config.PROCESSED_DIR)
# save_dir.mkdir(parents=True, exist_ok=True)  # Ensure directory exists

# # Paths for saving processed data
# train_pad_path = save_dir / "X_train_pad_dl.pkl"
# val_pad_path = save_dir / "X_val_pad_dl.pkl"
# test_sub_pad_path = save_dir / "X_test_sub_pad_dl.pkl"
# tokenizer_path = save_dir / "tokenizer_dl.pkl"

# # Save tokenized and padded sequences
# with open(train_pad_path, "wb") as f:
#     pickle.dump(X_train_pad, f)

# with open(val_pad_path, "wb") as f:
#     pickle.dump(X_val_pad, f)

# with open(test_sub_pad_path, "wb") as f:
#     pickle.dump(X_test_pad, f)  

# # Save the trained tokenizer
# with open(tokenizer_path, "wb") as f:
#     pickle.dump(tokenizer, f)

# # Print confirmation messages
# print(f"[✔] Tokenized & padded training dataset saved at: {train_pad_path}")
# print(f"[✔] Tokenized & padded validation dataset saved at: {val_pad_path}")
# print(f"[✔] Tokenized & padded test dataset saved at: {test_sub_pad_path}")
# print(f"[✔] Tokenizer saved at: {tokenizer_path}")

# Define save paths in the processed directory
save_dir = Path(config.PROCESSED_DIR)
save_dir.mkdir(parents=True, exist_ok=True)  # Ensure directory exists

# Paths for saving processed data
train_pad_path = save_dir / "X_train_pad_dl.pkl"
val_pad_path = save_dir / "X_val_pad_dl.pkl"
test_sub_pad_path = save_dir / "X_test_sub_pad_dl.pkl"
tokenizer_path = save_dir / "tokenizer_dl.pkl"

# Save tokenized and padded sequences using .to_pickle()
pd.to_pickle(X_train_pad, train_pad_path)
pd.to_pickle(X_val_pad, val_pad_path)
pd.to_pickle(X_test_pad, test_sub_pad_path)  # ✅ Plus besoin de convertir en DataFrame

# Save the trained tokenizer
pd.to_pickle(tokenizer, tokenizer_path)

# Print confirmation messages
print(f"[✔] Tokenized & padded training dataset saved at: {train_pad_path}")
print(f"[✔] Tokenized & padded validation dataset saved at: {val_pad_path}")
print(f"[✔] Tokenized & padded test dataset saved at: {test_sub_pad_path}")
print(f"[✔] Tokenizer saved at: {tokenizer_path}")



[✔] Tokenized & padded training dataset saved at: D:\Data_Science\Append_Data_Engineer_AWS_MLOPS\Data_Scientist_Rakuten_Project-main\data\processed\X_train_pad_dl.pkl
[✔] Tokenized & padded validation dataset saved at: D:\Data_Science\Append_Data_Engineer_AWS_MLOPS\Data_Scientist_Rakuten_Project-main\data\processed\X_val_pad_dl.pkl
[✔] Tokenized & padded test dataset saved at: D:\Data_Science\Append_Data_Engineer_AWS_MLOPS\Data_Scientist_Rakuten_Project-main\data\processed\X_test_sub_pad_dl.pkl
[✔] Tokenizer saved at: D:\Data_Science\Append_Data_Engineer_AWS_MLOPS\Data_Scientist_Rakuten_Project-main\data\processed\tokenizer_dl.pkl


## 4. 🔄Next Steps  

In this notebook, we have preprocessed the text data by tokenizing and padding the sequences, preparing them for deep learning models. The following processed data has been saved for future use:
- The **tokenizer** object, which contains the word-to-index mapping for tokenization.
- The **tokenized and padded sequences** for the training, test, and submission datasets.

---
### **Summary of Completed Steps**  
We have explored and processed textual data, from **initial data analysis** to **cleaning, visualization, and feature extraction** for both **Machine Learning and Deep Learning models**.  

Executed notebooks:  
- **01-03** → Data exploration (text & images).  
- **04** → Text cleaning & preprocessing.  
- **05** → Word Cloud visualization.  
- **06** → Data labeling & encoding.  
- **07** → TF-IDF vectorization for ML models.  
- **08** → Tokenization & sequence preparation for DL models.  

---
###  What’s Next?

In the next notebook, we will:

- Reproduce the **official CNN benchmark model** from the Rakuten Challenge  
- Use it to establish a strong baseline for text classification  
- Compare model performance **with and without text cleaning**

---

➡️ *Continue with the next notebook:*  
**`09_Benchmark_Text_Model.ipynb`**

